# OLD

In [1]:
import json 
import pandas as pd 
pd.options.mode.chained_assignment = None

In [2]:
with open("results/ideology_news_dichotomized/regression_results_with_scores_and_iterations.json") as file:
    res = pd.DataFrame(json.load(file)).T

res["reg-ID"] = [f"R-{i:06}" for i in range(len(res))]

print(f"Out of {len(res)} regressions, {100 * ('FAILED' != res['Coef']).mean():.0f}% were successful")
res_success = res.loc['FAILED' != res['Coef']]
print(f"out of {len(res_success)} successful regressions {100 * (res_success['LLR p-value'] <= 0.05).mean():.0f}% regressions are significant")
res_significative = res_success.loc[res_success["LLR p-value"] <= 0.05]
res_significative.loc[:,"bias"] = res_significative["x_column"].apply(lambda x : str(x).split("-")[0])
valid_for_comparison = res_significative.groupby(["bias", "y_column", "model", "learning_rate", "weight_decay"]).size().sort_values() == 2

print(f"out of {len(res_significative)} significant regressions, in only {100 * valid_for_comparison.mean():.0f}% of cases, we can compare the regression with the gold standard and predicted labels ({100 * valid_for_comparison.sum() / len(res):.0f}% of all regressions)")

def valid_ids(x: list):
    if len(x) == 2:
        return list(x)
valid_reg_ID = res_significative.groupby(["bias", "y_column", "model", "learning_rate", "weight_decay"], as_index=False)["reg-ID"].aggregate(valid_ids)["reg-ID"].to_list()

Out of 25488 regressions, 92% were significant
out of 23571 successful regressions 28% regressions are significant
out of 6640 significant regressions, in only 50% of cases, we can compare the regression with the gold standard and predicted labels (9% of all regressions)


In [3]:
IDS_for_errors_type_1 = []
IDS_for_errors_type_2 = []
IDS_for_errors_type_S = []
IDS_for_errors_type_M = []

N_configurations = sum([int(reg_IDs is not None) for reg_IDs in valid_reg_ID])

valid_reg_df = []

counter_valid_reg_group = 0

for reg_IDs in valid_reg_ID:
    if not reg_IDs:
        continue
    reg_cases = [
        res.loc[res["reg-ID"] == reg_IDs[0],:],
        res.loc[res["reg-ID"] == reg_IDs[1],:]
    ]
    if reg_cases[0]["x_column"].item().endswith("-GS"):
        id_GS, id_pred = 0, 1
    else:
        id_GS, id_pred = 1, 0
        
    GS_significant = reg_cases[id_GS]["pvalues"].item()[1] <= 0.05
    pred_significant = reg_cases[id_pred]["pvalues"].item()[1] <= 0.05

    GS_coef = reg_cases[id_GS]["Coef"].item()[1]
    pred_coef = reg_cases[id_pred]["Coef"].item()[1]

    error_type_1 = pred_significant and not GS_significant
    error_type_2 = GS_significant and not pred_significant
    error_type_S = pred_significant and GS_significant and (GS_coef * pred_coef < 0)

    if error_type_1: IDS_for_errors_type_1 += [reg_IDs]
    if error_type_2: IDS_for_errors_type_2 += [reg_IDs]
    if error_type_S: IDS_for_errors_type_S += [reg_IDs]

    valid_reg_df+= [
        {
            "reg-ID": reg_IDs[reg_cases_id],
            "valid-reg-group-ID" : f"valid-reg-{counter_valid_reg_group:06}",
            "error_type_1" : error_type_1,
            "error_type_2" : error_type_2,
            "error_type_S" : error_type_S,
        }
        for reg_cases_id in [0,1]
    ]
    counter_valid_reg_group += 1

print(f"IDS_for_errors_type_1: {len(IDS_for_errors_type_1)} / {N_configurations} ({100 * len(IDS_for_errors_type_1) / N_configurations:.0f}%)")
print(f"IDS_for_errors_type_2: {len(IDS_for_errors_type_2)} / {N_configurations} ({100 * len(IDS_for_errors_type_2) / N_configurations:.0f}%)")
print(f"IDS_for_errors_type_S: {len(IDS_for_errors_type_S)} / {N_configurations} ({100 * len(IDS_for_errors_type_S) / N_configurations:.0f}%)")

IDS_for_errors_type_1: 1129 / 2219 (51%)
IDS_for_errors_type_2: 9 / 2219 (0%)
IDS_for_errors_type_S: 2 / 2219 (0%)


In [4]:
res_success_with_error_type = res_success.set_index("reg-ID").join(pd.DataFrame(valid_reg_df).set_index("reg-ID"))
res_success_with_error_type.loc[:,["valid-reg-comparison"]] = res_success_with_error_type["valid-reg-group-ID"].notna()

In [9]:
import plotly.graph_objects as go
import numpy as np

col = "best_score"

x0 = (
    res_success_with_error_type
    .loc[
        res_success_with_error_type["valid-reg-group-ID"].notna(), 
        col
    ]
)

x1 = (
    res_success_with_error_type
    .loc[
        res_success_with_error_type["valid-reg-group-ID"].isna(), 
        col
    ]
)

fig = go.Figure()
fig.add_trace(go.Histogram(x=x0, name = "valid comparison"))
fig.add_trace(go.Histogram(x=x1, name = "invalid comparison"))

# Overlay both histograms
fig.update_layout(barmode='overlay')
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.show()


In [21]:
import plotly.express as px 


df_correlation = (
    res_success_with_error_type
    .loc[
        res_success_with_error_type["valid-reg-comparison"],
        [
            # "Pseudo R-squared", 
            "LLR p-value", 
            # "N iterations", 
            "Score on sample", 
            # "best_score", 
            "error_type_1", 
            "error_type_2", 
            "error_type_S", 
        ]
    ]
)
px.imshow(df_correlation.corr(), color_continuous_scale='RdBu_r',color_continuous_midpoint = 0, range_color = [-0.5, 0.5])

# New

In [183]:
import json 
import numpy as np 
import pandas as pd 
pd.options.mode.chained_assignment = None

In [194]:
with open("./results/ideology_news_dichotomized/answerdotai/ModernBERT-base/ideology_news_dichotomized-regression-results.json") as file:
    res_1 = pd.DataFrame(json.load(file)).T
with open("./results/ideology_news_dichotomized/ibm-granite/granite-embedding-small-english-r2-regression-results.json") as file:
    res_2 = pd.DataFrame(json.load(file)).T
res = pd.concat((res_1, res_2))

# reindex = 
res.loc[:,"R-ID"] = [f"R-{i:06}" for i in range(len(res))]

res_success = res.loc['FAILED' != res['Coef']]

In [197]:
res

,R-ID,task,hypothesis,label-type,model,label_dichotomized,learning_rate,weight_decay,best_epoch,best_score,...,z,pvalues,Conf Int,Log-Likelihood,LL-Null,LLR p-value,AIC,BIC,N obs,N iterations
0,R-000000,ideology_news_dichotomized-bias_left,bias_left~T-middle_east,GS,answerdotai/ModernBERT-base,bias_left,0.00001,0.05,4,0.73575,...,"[-43.325091955691406, -1.0383212251100662]","[0.0, 0.2991205303150526]","[[-1.0309553777697624, -0.9417146616817216], [...",-5840.966775,-5841.519471,0.293086,11685.93355,11700.354231,10000,5
1,R-000001,ideology_news_dichotomized-bias_left,bias_left~T-justice_department,GS,answerdotai/ModernBERT-base,bias_left,0.00001,0.05,4,0.73575,...,"[-43.79137436062213, -0.7324308970731194]","[0.0, 0.4639056061341089]","[[-1.032894626960244, -0.9443972696416867], [-...",-5841.241745,-5841.519471,0.456098,11686.48349,11700.90417,10000,5
2,R-000002,ideology_news_dichotomized-bias_left,bias_left~T-impeachment,GS,answerdotai/ModernBERT-base,bias_left,0.00001,0.05,4,0.73575,...,"[-43.65931492477044, -1.8483587473630414]","[0.0, 0.06455046230409965]","[[-1.0302808441747633, -0.9417518977809204], [...",-5839.633834,-5841.519471,0.05214,11683.267668,11697.688349,10000,5
3,R-000003,ideology_news_dichotomized-bias_left,bias_left~T-white_house,GS,answerdotai/ModernBERT-base,bias_left,0.00001,0.05,4,0.73575,...,"[-42.863486863796474, -0.3754138268276222]","[0.0, 0.7073527233148758]","[[-1.0333453221212834, -0.9429765719862937], [...",-5841.448605,-5841.519471,0.706565,11686.897209,11701.31789,10000,5
4,R-000004,ideology_news_dichotomized-bias_left,bias_left~T-healthcare,GS,answerdotai/ModernBERT-base,bias_left,0.00001,0.05,4,0.73575,...,"[-43.48088311619855, 2.9849328348877813]","[0.0, 0.0028364059635417023]","[[-1.0501195492902553, -0.9595316035370661], [...",-5837.210639,-5841.519471,0.003329,11678.421278,11692.841958,10000,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25483,R-050971,ideology_news_dichotomized-bias_right,bias_right~S-AZ Central,PRED,ibm-granite/granite-embedding-small-english-r2,bias_right,0.0005,0.01,4,0.795111,...,"[-24.29603670697818, -0.0014567202397687646]","[2.1588393053439557e-130, 0.9988377058223508]","[[-0.5417283756277804, -0.4608502466157843], [...",-6626.293812,-6626.767372,0.330453,13256.587624,13271.008305,10000,100
25484,R-050972,ideology_news_dichotomized-bias_right,bias_right~S-San Jose Mercury News,PRED,ibm-granite/granite-embedding-small-english-r2,bias_right,0.0005,0.01,4,0.795111,...,"[-24.29603670697818, -0.0014567202397687646]","[2.1588393053439557e-130, 0.9988377058223508]","[[-0.5417283756277804, -0.4608502466157843], [...",-6626.293812,-6626.767372,0.330453,13256.587624,13271.008305,10000,100
25485,R-050973,ideology_news_dichotomized-bias_right,bias_right~S-MSNBC,PRED,ibm-granite/granite-embedding-small-english-r2,bias_right,0.0005,0.01,4,0.795111,...,"[-24.29603670697818, -0.0014567202397687646]","[2.1588393053439557e-130, 0.9988377058223508]","[[-0.5417283756277804, -0.4608502466157843], [...",-6626.293812,-6626.767372,0.330453,13256.587624,13271.008305,10000,100
25486,R-050974,ideology_news_dichotomized-bias_right,bias_right~S-Concord Monitor,PRED,ibm-granite/granite-embedding-small-english-r2,bias_right,0.0005,0.01,4,0.795111,...,"[-24.29603670697818, -0.0014567202397687646]","[2.1588393053439557e-130, 0.9988377058223508]","[[-0.5417283756277804, -0.4608502466157843], [...",-6626.293812,-6626.767372,0.330453,13256.587624,13271.008305,10000,100


In [198]:
# Retrieve valid duo of regressions and create an index
def return_valid_duo(x: pd.Series):
    if len(x) == 2:
        return list(x)
    else:
        return "INVALID"

list_of_valid_duo = list(
    res_success
    .groupby(["task", "hypothesis", "model", "learning_rate", "weight_decay"])
    ["R-ID"]
    .agg(return_valid_duo)
    .values
)

counter_duo_reg = 0
duo_index = pd.Series(dtype=str)
for duo in list_of_valid_duo: 
    if duo == "INVALID":
        continue 

    duo_index[duo[0]] = f"RegDuo-{counter_duo_reg:06}"
    duo_index[duo[1]] = f"RegDuo-{counter_duo_reg:06}"
    counter_duo_reg += 1

res_success = res_success.set_index("R-ID").join(pd.DataFrame({"RDuo-ID": duo_index}))
res_success_only_duo = res_success.loc[res_success["RDuo-ID"].notna()]

In [232]:
# For each duo test error types
def evaluate_errors(sub_df : pd.DataFrame):
    
    index_covariate = sub_df["Covariate Names"].iloc[0].index("x1")

    significant_GS : bool = (
        sub_df.loc[
            sub_df["label-type"] == "GS",
            "pvalues"
        ]
        .item()
        [index_covariate]
        < 
        0.05
    )
    significant_PRED : bool = (
        sub_df.loc[
            sub_df["label-type"] == "PRED",
            "pvalues"
        ]
        .item()
        [index_covariate]
        < 
        0.05
    )
    coef_GS_is_positive : bool = (
        sub_df.loc[
            sub_df["label-type"] == "PRED",
            "Coef"
        ]
        .item()
        [index_covariate]
        > 
        0
    )
    coef_PRED_is_positive : bool = (
        sub_df.loc[
            sub_df["label-type"] == "PRED",
            "Coef"
        ]
        .item()
        [index_covariate]
        > 
        0
    )

    return {
        "RDuo-ID": sub_df["RDuo-ID"].iloc[0],
        "R-IDs": sub_df.index.to_list(),
        "significant_GS" : significant_GS,
        "significant_PRED" : significant_PRED,
        "coef_GS_is_positive" : coef_GS_is_positive,
        "coef_PRED_is_positive" : coef_PRED_is_positive,
    }

error_tables = pd.DataFrame(res_success_only_duo.groupby("RDuo-ID").apply(evaluate_errors).to_list())
error_tables["type-I"] = np.logical_and(~error_tables["significant_GS"], error_tables["significant_PRED"])
error_tables["type-II"] = np.logical_and(error_tables["significant_GS"], ~error_tables["significant_PRED"])
error_tables["type-S"] = np.logical_and.reduce((
    error_tables["significant_GS"],
    error_tables["significant_PRED"],
    error_tables["coef_GS_is_positive"] != error_tables["coef_PRED_is_positive"]
))

/var/folders/xs/d90v1vkn16db_7z493h6kwt40000gn/T/ipykernel_3151/1714523123.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  error_tables = pd.DataFrame(res_success_only_duo.groupby("RDuo-ID").apply(evaluate_errors).to_list())


In [234]:
print(f" Error type I : {error_tables['type-I'].mean() * 100:.0f} % ({error_tables['type-I'].sum()} / {len(error_tables)})")
print(f"Error type II : {error_tables['type-II'].mean() * 100:.0f} % ({error_tables['type-II'].sum()} / {len(error_tables)})")
print(f" Error type S : {error_tables['type-S'].mean() * 100:.0f} % ({error_tables['type-S'].sum()} / {len(error_tables)})")

 Error type I : 10 % (2370 / 22898)
Error type II : 2 % (439 / 22898)
 Error type S : 0 % (0 / 22898)


In [237]:
# Evaluate "RISK" for error type I
table = []
for RDuoID in error_tables["RDuo-ID"].unique():
    res_row = res_success_only_duo[res_success_only_duo["RDuo-ID"] == RDuoID].iloc[0]
    error_row = error_tables[error_tables["RDuo-ID"] == RDuoID].iloc[0]
    task = res_row["task"]
    H_t = int(error_row["significant_GS"])
    config = "-".join([f"{key}:{res_row[key]}" for key in ["model", "learning_rate", "weight_decay"]])
    error_type_I = error_row["type-I"]
    error_type_II = error_row["type-II"]
    error_type_S = error_row["type-S"]
    table+=[{
        "task" :            task,
        "H_t" :             H_t,
        "config" :           config,
        "error_type_I" :    error_type_I,
        "error_type_II" :   error_type_II,
        "error_type_S" :    error_type_S,
    }]

aggregated_table = pd.DataFrame(table)
aggregated_table

,task,H_t,config,error_type_I,error_type_II,error_type_S
0,ideology_news_dichotomized-bias_center,0,model:answerdotai/ModernBERT-base-learning_rat...,False,False,False
1,ideology_news_dichotomized-bias_center,0,model:answerdotai/ModernBERT-base-learning_rat...,False,False,False
2,ideology_news_dichotomized-bias_center,0,model:answerdotai/ModernBERT-base-learning_rat...,False,False,False
3,ideology_news_dichotomized-bias_center,0,model:answerdotai/ModernBERT-base-learning_rat...,False,False,False
4,ideology_news_dichotomized-bias_center,0,model:answerdotai/ModernBERT-base-learning_rat...,False,False,False
...,...,...,...,...,...,...
22893,ideology_news_dichotomized-bias_right,1,model:ibm-granite/granite-embedding-small-engl...,False,False,False
22894,ideology_news_dichotomized-bias_right,1,model:ibm-granite/granite-embedding-small-engl...,False,False,False
22895,ideology_news_dichotomized-bias_right,1,model:ibm-granite/granite-embedding-small-engl...,False,False,False
22896,ideology_news_dichotomized-bias_right,1,model:ibm-granite/granite-embedding-small-engl...,False,False,False


In [241]:
risk_I = 0
risk_II = 0
risk_S = 0

T = aggregated_table["task"].nunique()
PHI = aggregated_table["config"].nunique()
print(f"T : {T}")
print(f"PHI : {PHI}")

for (task, config), sub_df in aggregated_table.groupby(["task", "config"]):
    risk_I  += sub_df.loc[sub_df["H_t"] == 0 , "error_type_I" ].mean()
    risk_II += sub_df.loc[sub_df["H_t"] == 1, "error_type_II"].mean()
    risk_S  += sub_df.loc[sub_df["H_t"] == 1, "error_type_S"].mean()
risk_I =  risk_I  / (T * PHI)
risk_II = risk_II / (T * PHI)
risk_S =  risk_S  / (T * PHI)

print(f" Risk I : {100 * risk_I:.1f}")
print(f"Risk II : {100 * risk_II:.1f}")
print(f" Risk S : {100 * risk_S:.1f}")

T : 3
PHI : 24
 Risk I : 11.0
Risk II : 31.8
 Risk S : 0.0
